# AB: pooled direct logistic validation

One model per lead; all oblasts pooled, no identity encoding, calendar or STL predictors. Leads 1–6 are primary; lead 0 is separate; 12/24 are secondary. Each fold starts from scratch on TRAIN ending 25 hours before VALID. Scaling, rates and coefficients fit TRAIN only. C is selected by pooled VALID Brier over four independent folds; these are selection scores, not unbiased final performance. No resampling or TEST evaluation.


In [1]:
from pathlib import Path
import sys
import pandas as pd
root = Path.cwd() if Path("src/airalerts").exists() else Path.cwd().parent
sys.path.insert(0, str(root / "src"))
from airalerts.evaluate import latest_results
out = root / "reports/model"
latest = latest_results(out / "results.csv")


In [2]:
results = pd.read_csv(out / "AB_validation.csv", dtype={"fold_id": str})
summary = results.loc[results.fold_id.eq("pooled") & results.scope.eq("ALL")]
print(summary[["model", "lead", "n", "brier", "bss_vs_clim", "bss_vs_persistence", "bss_vs_ses", "bss_ci_low", "bss_ci_high", "ece"]].to_string(index=False))
print(pd.read_csv(out / "AB_selected.csv").to_string(index=False))


              model  lead     n    brier  bss_vs_clim  bss_vs_persistence  bss_vs_ses  bss_ci_low  bss_ci_high      ece
          logreg_AB     0 35136 0.054814     0.438425            0.042692    0.348861    0.409243     0.470069 0.013353
logreg_AB_neighbors     0 35136 0.054746     0.439123            0.043883    0.349671    0.409730     0.469569 0.011379
 logreg_AB_national     0 35136 0.055029     0.436218            0.038929    0.346302    0.408761     0.465936 0.012162
     logreg_AB_base     0 35136 0.054988     0.436640            0.039649    0.346791    0.408552     0.465724 0.015377
          logreg_AB     1 35112 0.087916     0.099399            0.041897    0.044567    0.065505     0.139226 0.029058
logreg_AB_neighbors     1 35112 0.088312     0.095342            0.037581    0.040262    0.062184     0.134411 0.027633
 logreg_AB_national     1 35112 0.088282     0.095659            0.037918    0.040599    0.064128     0.132366 0.027080
     logreg_AB_base     1 35112 0.088642

In [3]:
print(summary.pivot(index="lead", columns="model", values="bss_vs_clim").to_string())
print(pd.read_csv(out / "ab_selection.csv").to_string(index=False))


model  logreg_AB  logreg_AB_base  logreg_AB_national  logreg_AB_neighbors
lead                                                                     
0       0.438425        0.436640            0.436218             0.439123
1       0.099399        0.091964            0.095659             0.095342
2       0.065196        0.057479            0.060291             0.059779
3       0.057834        0.050683            0.051927             0.053229
4       0.048987        0.047273            0.043308             0.046335
5       0.043766        0.041901            0.037528             0.041208
6       0.039813        0.036251            0.032428             0.036417
12      0.025577        0.024368            0.017363             0.021632
24      0.016170        0.017413            0.009892             0.011201
 lead  relative_brier_gain_vs_A    ci_low  ci_high  keep_context                   decision           run_id
    0                  0.004023 -0.006054 0.013975         False no evidence 

## Calibration

ECE uses ten bins. CIs pair all regions within circular seven-day UTC blocks; one-day blocks are sensitivity only. Flags are computed separately from each fold TRAIN target rate, below 3% or above 95%; per-oblast and pooled excluded-flagged metrics are in results.csv.


![Calibration: AB](../reports/model/AB_calibration.png)

The paired seven-day gain interval excludes zero in favor of full context for leads [1, 2, 3, 4, 5, 6]. For leads [0, 12, 24], there is no evidence of improvement and Group B is not carried into recency. Neighbor-only, national-only and base-rate-only additions are descriptive ablations in the table; no geographic identity predictor is added.
